In [2]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, Dense
from tensorflow.keras.utils import to_categorical

In [3]:
with open("input_1.txt", "r", encoding="utf-8") as file:
    text = file.read()

print("Sample text:")
print(text[:528])

Sample text:
Once upon a time there was a quiet village surrounded by green hills and thick forests. The village was small but beautiful. People lived simple lives and helped each other every day. The mornings were peaceful and the air was always fresh. Children played in the fields while farmers worked hard to grow crops. Everyone in the village respected nature and lived in harmony with the forest nearby.

In that village lived a young boy named Arin. Arin was curious and adventurous. He loved asking questions and exploring new place


In [4]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])

total_words = len(tokenizer.word_index) + 1

print("Total words:", total_words)

Total words: 347


The Tokenizer scans the entire text and assigns a unique index number to each word.

The dictionary tokenizer.word_index stores the mapping of words to numerical indices.

total_words represents the size of the vocabulary.

This step converts textual data into numerical form so that it can be processed by neural networks.

In [5]:
input_sequences = []

for line in text.split("\n"):
    token_list = tokenizer.texts_to_sequences([line])[0]

    for i in range(1, len(token_list)):
        n_gram = token_list[:i+1]
        input_sequences.append(n_gram)

max_seq_len = max([len(seq) for seq in input_sequences])

input_sequences = np.array(
    pad_sequences(input_sequences, maxlen=max_seq_len, padding='pre')
)

X = input_sequences[:, :-1]
y = input_sequences[:, -1]

y = to_categorical(y, num_classes=total_words)

This cell generates input sequences for next-word prediction.

Each line in the text is converted into a sequence of tokens.

Then n-gram sequences are created.

These sequences help the model learn the relationship between words in a sentence.



>Padding ensures that all sequences have equal length.

>The sequences are split into:

>X (input words)

>y (next word to predict)

>The output word y is converted into one-hot encoded format using to_categorical.

>This makes the dataset suitable for training the neural network.

In [6]:
rnn_model = Sequential([
    Embedding(total_words, 100, input_length=max_seq_len-1),
    SimpleRNN(150),
    Dense(total_words, activation='softmax')
])

rnn_model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

rnn_model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

This cell constructs the Recurrent Neural Network (RNN) model.

The architecture consists of:

Embedding Layer – Converts words into dense vector representations.

SimpleRNN Layer – Processes sequential data and learns temporal relationships.

Dense Layer with Softmax – Predicts the probability distribution of the next word.

Categorical Crossentropy is used as the loss function.

Adam optimizer updates the model weights efficiently.

Accuracy is used as a performance metric.

In [7]:
rnn_model.fit(X, y, epochs=50, verbose=1)

Epoch 1/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 4s 58ms/step - accuracy: 0.0416 - loss: 5.6689
Epoch 2/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 45ms/step - accuracy: 0.1047 - loss: 5.1130
Epoch 3/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step - accuracy: 0.1177 - loss: 4.9800
Epoch 4/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 33ms/step - accuracy: 0.1020 - loss: 4.9831
Epoch 5/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step - accuracy: 0.1172 - loss: 4.8479
Epoch 6/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step - accuracy: 0.1228 - loss: 4.6141
Epoch 7/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 33ms/step - accuracy: 0.1552 - loss: 4.3677
Epoch 8/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step - accuracy: 0.1896 - loss: 4.0556
Epoch 9/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step - accuracy: 0.2604 - loss: 3.7949
Epoch 10/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 33ms/step - accuracy: 0.3272 - loss: 3.4709
Epoch 11/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 33ms/step - accuracy: 0.3898 - loss: 3.0553
Epoch 12/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 49ms/step - accuracy:

This cell trains the RNN model using the prepared dataset.

X contains input word sequences.

y contains the target next word.

Epochs = 50 means the model learns from the dataset 50 times.

In [8]:
lstm_model = Sequential([
    Embedding(total_words, 100, input_length=max_seq_len-1),
    LSTM(150),
    Dense(total_words, activation='softmax')
])

lstm_model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

lstm_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

This cell constructs the LSTM (Long Short-Term Memory) model.

LSTM is an advanced version of RNN designed to overcome the vanishing gradient problem and capture long-term dependencies in sequences.

The model includes:

>Embedding layer

>LSTM layer

>Dense output layer

The model learns patterns in the text and improves its ability to predict the next word.

In [9]:
lstm_model.fit(X, y, epochs=50, verbose=1)

Epoch 1/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 7s 131ms/step - accuracy: 0.0694 - loss: 5.7315
Epoch 2/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 3s 116ms/step - accuracy: 0.1042 - loss: 5.1738
Epoch 3/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 3s 108ms/step - accuracy: 0.1110 - loss: 5.0617
Epoch 4/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 4s 162ms/step - accuracy: 0.1097 - loss: 4.9515
Epoch 5/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 3s 107ms/step - accuracy: 0.1028 - loss: 5.0576
Epoch 6/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 3s 109ms/step - accuracy: 0.1112 - loss: 4.9370
Epoch 7/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 3s 109ms/step - accuracy: 0.1102 - loss: 4.8203
Epoch 8/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 5s 165ms/step - accuracy: 0.1180 - loss: 4.7530
Epoch 9/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 3s 108ms/step - accuracy: 0.1495 - loss: 4.5906
Epoch 10/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 5s 109ms/step - accuracy: 0.1632 - loss: 4.4251
Epoch 11/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 5s 165ms/step - accuracy: 0.1651 - loss: 4.3231
Epoch 12/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 3s 108ms/step

In [ ]:
def generate_text(model, seed_text, next_words):

    for _ in range(next_words):

        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_seq_len-1, padding='pre')

        predicted = np.argmax(model.predict(token_list), axis=-1)

        output_word = ""

        for word, index in tokenizer.word_index.items():
            if index == predicted:
                output_word = word
                break

        seed_text += " " + output_word

    return seed_text

In [ ]:
seed_text = "artificial intelligence"

print("RNN Generated Text:")
print(generate_text(rnn_model, seed_text, 10))

print("\nLSTM Generated Text:")
print(generate_text(lstm_model, seed_text, 10))

RNN Generated Text:
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 158ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
artificial intelligence was curious and mysterious him in the forest had secrets

LSTM Generated Text:
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 173ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
artificial intelligence bright morning the old man gave him a map should


A function generate_text() is used to generate new text using the trained model.

The seed text "artificial intelligence" is given as the starting input.

The model predicts the next word based on the learned patterns from the training data.

Each predicted word is added to the sentence step by step.

This process continues until 10 new words are generated.

The function is executed for both RNN and LSTM models.

The generated sentences are printed as output.

This step shows that the trained models can generate text similar to the training dataset.